# 1. Configuração Inicial e Imports

## Objetivo desta Seção
Importar todas as bibliotecas necessárias para a solução completa e configurar parâmetros visuais dos gráficos.

## Bibliotecas Utilizadas

### Processamento e Estruturas de Dados
- **networkx (nx)**: Criação, manipulação e análise de grafos
- **pandas (pd)**: Estruturas de dados para análise (DataFrame, Series)
- **numpy (np)**: Computação numérica e operações matemáticas

### Visualização
- **matplotlib.pyplot (plt)**: Criação de gráficos e visualizações
- **seaborn (sns)**: Estilo e temas para matplotlib

### Utilitários
- **os**: Operações do sistema operacional
- **pathlib.Path**: Manipulação de caminhos de arquivos
- **time**: Medição de tempo de execução
- **itertools.product**: Geração de combinações
- **typing**: Type hints para melhor documentação

## Configurações Visuais
- Estilo whitegrid para melhor legibilidade
- Figura padrão de 12x6 polegadas
- Tamanho de fonte padrão de 10pt

In [2]:
# Importar todas as bibliotecas necessárias
import os
import csv
import time
import json
import random
from pathlib import Path
from itertools import product
from typing import Dict, Tuple, List, Set

import networkx as nx
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Configurar estilo dos gráficos
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("✓ Todas as bibliotecas importadas com sucesso")

✓ Todas as bibliotecas importadas com sucesso


# 2. Criação da Estrutura de Diretórios

## Objetivo desta Seção
Implementar uma classe auxiliar para criar e gerenciar automaticamente a estrutura de diretórios necessária para armazenar resultados, gráficos e visualizações de ambas as partes da solução.

## Estrutura Criada

```
resultados/
├── parte1/
│   ├── csv/              # Parâmetros e resultados da força bruta
│   ├── grafos/           # Visualizações dos grafos coloridos (PNG)
│   └── graficos/         # Gráficos de análise e escalabilidade (PNG)
└── parte2/
    ├── csv/              # Resultados da heurística DIMACS
    ├── grafos/           # Visualizações dos grafos coloridos (PNG)
    └── graficos/         # Gráficos comparativos (PNG)
```

## Funcionalidades da Classe ConfiguradorDiretorios

### `criar_estrutura()`
- Cria automaticamente toda a estrutura de diretórios
- Garante que todos os caminhos existem antes de salvar arquivos
- Usa `parents=True` e `exist_ok=True` para evitar erros

### `obter_caminho_arquivo(tipo_parte, tipo_saida, nome_arquivo)`
- Fornece caminhos consistentes para arquivo de saída
- Exemplo: `'resultados_1/parte1/csv/parametros_grafos.csv'`
- Reduz hardcoding de caminhos no código

In [3]:
class ConfiguradorDiretorios:
    """Classe auxiliar para criar e gerenciar estrutura de diretórios."""
    
    ESTRUTURA_DIRETORIOS = {
        'resultados_1/parte1/csv': [],
        'resultados_1/parte1/grafos': [],
        'resultados_1/parte1/graficos': [],
        'resultados_1/parte2/csv': [],
        'resultados_1/parte2/grafos': [],
        'resultados_1/parte2/graficos': [],
    }
    
    @staticmethod
    def criar_estrutura():
        """Cria a estrutura de diretórios completa."""
        for diretorio in ConfiguradorDiretorios.ESTRUTURA_DIRETORIOS.keys():
            Path(diretorio).mkdir(parents=True, exist_ok=True)
        print(f"✓ Estrutura de diretórios criada com sucesso")
    
    @staticmethod
    def obter_caminho_arquivo(tipo_parte: str, tipo_saida: str, nome_arquivo: str) -> str:
        """
        Retorna caminho completo para arquivo baseado em tipo.
        
        Args:
            tipo_parte: 'parte1' ou 'parte2'
            tipo_saida: 'csv', 'grafos' ou 'graficos'
            nome_arquivo: nome do arquivo
        
        Returns:
            Caminho completo do arquivo
        """
        return f"resultados_1/{tipo_parte}/{tipo_saida}/{nome_arquivo}"

# Criar estrutura de diretórios
ConfiguradorDiretorios.criar_estrutura()

✓ Estrutura de diretórios criada com sucesso


# 3. Funções Auxiliares Comuns

## Objetivo desta Seção
Definir funções auxiliares compartilhadas entre Parte 1 (Força Bruta) e Parte 2 (Heurística) para evitar duplicação de código.

## Funções Implementadas

### 1. `extrair_parametros_grafo(grafo)`
Extrai métricas estatísticas de um grafo NetworkX.

**Parâmetros Extraídos:**
- `num_vertices`: Número de vértices
- `num_arestas`: Número de arestas
- `densidade`: Razão arestas/(vértices²), indica "compactação" do grafo
- `grau_medio`: Média dos graus de todos os vértices
- `grau_minimo`: Menor grau encontrado
- `grau_maximo`: Maior grau encontrado
- `eh_conexo`: Boolean se grafo é conexo
- `num_componentes`: Número de componentes conectadas

**Uso:** Coletar informações para análise posterior

---

### 2. `visualizar_e_salvar_grafo(grafo, coloracao, caminho_saida)`
Cria visualizações de grafos com coloração e salva como PNG.

**Características:**
- Layout spring (force-directed) para melhor disposição visual
- Colormap 'Set3' com cores distintas por número cromático
- Arestas com alpha=0.3 para não obscurecer nós
- Título mostra χ(G) encontrado
- Salva em alta resolução (150 dpi)

**Uso:** Documentar visualmente a coloração encontrada

---

### 3. `ler_arquivo_dimacs(caminho_arquivo)`
Parser para o formato DIMACS padrão de competições de coloração.

**Formato DIMACS:**
```
c comentário
c comentário
p edge num_vertices num_arestas
e vertice1 vertice2
e vertice3 vertice4
...
```

**Características:**
- Ignora linhas de comentário ('c')
- Processa linha 'p edge' para dimensões
- Processa linhas 'e' para arestas
- Tratamento de erro se arquivo não existir
- Retorna grafo NetworkX ou None

**Uso:** Ler 5 instâncias do Moodle na Parte 2

In [4]:
def extrair_parametros_grafo(grafo: nx.Graph) -> Dict:
    """
    Extrai parâmetros estatísticos de um grafo.
    
    Função auxiliar comum usada em ambas as partes para extrair informações
    de um grafo (densidade, grau médio, conectividade, etc.).
    
    Args:
        grafo: Grafo NetworkX
        
    Returns:
        Dicionário com parâmetros do grafo
    """
    return {
        'num_vertices': grafo.number_of_nodes(),
        'num_arestas': grafo.number_of_edges(),
        'densidade': nx.density(grafo),
        'grau_medio': sum(dict(grafo.degree()).values()) / grafo.number_of_nodes(),
        'grau_minimo': min(dict(grafo.degree()).values()),
        'grau_maximo': max(dict(grafo.degree()).values()),
        'eh_conexo': nx.is_connected(grafo),
        'num_componentes': nx.number_connected_components(grafo),
    }

print("✓ Função extrair_parametros_grafo definida")

✓ Função extrair_parametros_grafo definida


In [5]:
def ler_arquivo_dimacs(caminho_arquivo: str) -> nx.Graph:
    """
    Função auxiliar comum: Lê arquivo DIMACS e cria grafo NetworkX.
    
    Formato DIMACS:
    - Linhas começadas com 'c' são comentários
    - Linha 'p edge <n> <m>' define n vértices e m arestas
    - Linhas 'e <u> <v>' definem arestas
    
    Args:
        caminho_arquivo: Caminho do arquivo DIMACS
        
    Returns:
        Grafo NetworkX ou None se arquivo não existir
    """
    if not os.path.exists(caminho_arquivo):
        print(f"⚠ Arquivo não encontrado: {caminho_arquivo}")
        return None
    
    grafo = nx.Graph()
    
    try:
        with open(caminho_arquivo, 'r', encoding='utf-8', errors='ignore') as f:
            for linha in f:
                linha = linha.strip()
                
                # Ignorar comentários
                if linha.startswith('c'):
                    continue
                
                # Linha de problema (define número de vértices)
                if linha.startswith('p edge'):
                    partes = linha.split()
                    num_vertices = int(partes[2])
                    num_arestas = int(partes[3])
                    grafo.add_nodes_from(range(1, num_vertices + 1))
                    continue
                
                # Linhas de aresta
                if linha.startswith('e'):
                    partes = linha.split()
                    u = int(partes[1])
                    v = int(partes[2])
                    grafo.add_edge(u, v)
        
        return grafo
    
    except Exception as e:
        print(f"⚠ Erro ao ler arquivo DIMACS: {str(e)}")
        return None

print("✓ Função ler_arquivo_dimacs definida")

✓ Função ler_arquivo_dimacs definida


In [6]:
def visualizar_e_salvar_grafo(grafo: nx.Graph, coloracao: Dict, caminho_saida: str):
    """
    Visualiza grafo com coloração e salva como PNG.
    
    Função auxiliar comum que cria visualização dos grafos coloridos em ambas
    as partes da solução. Adapta a visualização conforme o tamanho do grafo.
    Implementa visualização adaptativa com estatísticas visuais.
    
    Args:
        grafo: Grafo NetworkX
        coloracao: Dicionário {vertice: cor}
        caminho_saida: Caminho para salvar arquivo PNG
    """
    num_vertices = grafo.number_of_nodes()
    num_arestas = grafo.number_of_edges()
    cores_set = set(coloracao.values())
    num_cores = max(cores_set) + 1
    cores_lista = [coloracao[node] for node in grafo.nodes()]
    densidade = nx.density(grafo)
    
    # Calcular distribuição de cores
    distribuicao_cores = {}
    for cor in cores_set:
        distribuicao_cores[cor] = sum(1 for v in coloracao.values() if v == cor)
    
    # Criar figura com subplot para grafos grandes (inclui estatísticas)
    if num_vertices <= 100:
        # ======= GRAFOS MUITO PEQUENOS (≤100): Visualização Completa com Labels =======
        fig = plt.figure(figsize=(12, 8))
        
        # Subplot principal: grafo
        ax_grafo = plt.subplot(1, 2, 1)
        pos = nx.spring_layout(grafo, seed=42, iterations=50)
        
        nx.draw_networkx_edges(grafo, pos, alpha=0.3, width=0.8, ax=ax_grafo)
        nx.draw_networkx_nodes(grafo, pos, node_color=cores_lista, 
                              node_size=500, cmap='Set3', vmin=0, 
                              vmax=max(cores_set), ax=ax_grafo)
        nx.draw_networkx_labels(grafo, pos, font_size=7, ax=ax_grafo)
        
        ax_grafo.set_title(f'Coloração de Vértices\n{num_vertices} vértices, {num_arestas} arestas',
                          fontsize=12, fontweight='bold')
        ax_grafo.axis('off')
        
        # Subplot de estatísticas: distribuição de cores
        ax_stats = plt.subplot(1, 2, 2)
        cores_ids = sorted(distribuicao_cores.keys())
        contagens = [distribuicao_cores[c] for c in cores_ids]
        colors_bar = plt.cm.Set3(np.linspace(0, 1, num_cores))
        
        bars = ax_stats.bar(cores_ids, contagens, color=colors_bar[:len(cores_ids)], 
                            edgecolor='black', linewidth=1.5)
        ax_stats.set_xlabel('Cor', fontsize=11, fontweight='bold')
        ax_stats.set_ylabel('Número de Vértices', fontsize=11, fontweight='bold')
        ax_stats.set_title('Distribuição de Cores', fontsize=12, fontweight='bold')
        ax_stats.grid(True, alpha=0.3, axis='y')
        
        # Adicionar valores nas barras
        for bar, count in zip(bars, contagens):
            height = bar.get_height()
            ax_stats.text(bar.get_x() + bar.get_width()/2., height,
                         f'{int(count)}', ha='center', va='bottom', fontweight='bold')
        
        # Adicionar informações estatísticas
        info_text = f"χ(G) = {num_cores} cores\nDensidade: {densidade:.4f}\nGrau médio: {2*num_arestas/num_vertices:.2f}"
        ax_stats.text(0.02, 0.98, info_text, transform=ax_stats.transAxes,
                     fontsize=10, verticalalignment='top',
                     bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
        
        dpi = 150
        
    elif num_vertices <= 1000:
        # ======= GRAFOS PEQUENOS/MÉDIOS (101-1000): Layout Otimizado + Stats =======
        # Inclui instâncias a (450) e b (864) e c (1000)
        fig = plt.figure(figsize=(14, 6))
        
        # Subplot principal: grafo
        ax_grafo = plt.subplot(1, 3, (1, 2))
        try:
            pos = nx.kamada_kawai_layout(grafo)
        except:
            pos = nx.spring_layout(grafo, seed=42, iterations=30, k=0.15)
        
        nx.draw_networkx_edges(grafo, pos, alpha=0.15, width=0.3, ax=ax_grafo)
        nx.draw_networkx_nodes(grafo, pos, node_color=cores_lista,
                              node_size=150, cmap='Set3', vmin=0,
                              vmax=max(cores_set), ax=ax_grafo, edgecolors='black', linewidths=0.5)
        
        ax_grafo.set_title(f'Coloração de Vértices - {num_vertices} vértices\nχ(G) = {num_cores} cores',
                          fontsize=12, fontweight='bold')
        ax_grafo.axis('off')
        
        # Subplot de estatísticas
        ax_stats = plt.subplot(1, 3, 3)
        cores_ids = sorted(distribuicao_cores.keys())
        contagens = [distribuicao_cores[c] for c in cores_ids]
        colors_bar = plt.cm.Set3(np.linspace(0, 1, num_cores))
        
        ax_stats.barh(cores_ids, contagens, color=colors_bar[:len(cores_ids)],
                     edgecolor='black', linewidth=1)
        ax_stats.set_ylabel('Cor', fontsize=10, fontweight='bold')
        ax_stats.set_xlabel('Vértices', fontsize=10, fontweight='bold')
        ax_stats.set_title('Distribuição', fontsize=11, fontweight='bold')
        ax_stats.grid(True, alpha=0.3, axis='x')
        ax_stats.invert_yaxis()
        
        info_text = f"Vértices: {num_vertices}\nArestas: {num_arestas}\nχ(G): {num_cores}\nDensidade: {densidade:.5f}\nGrau médio: {2*num_arestas/num_vertices:.2f}"
        ax_stats.text(0.02, 0.02, info_text, transform=ax_stats.transAxes,
                     fontsize=9, verticalalignment='bottom',
                     bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))
        
        dpi = 120
        
    else:
        # ======= GRAFOS GRANDES (>1000): Visualização Simplificada Consistente =======
        fig = plt.figure(figsize=(14, 6))
        
        # Subplot 1: Grafo simplificado (ocupa 2/3 do espaço)
        ax_grafo = plt.subplot(1, 3, (1, 2))
        try:
            # Para grafos muito grandes, usar layout mais rápido
            pos = nx.spring_layout(grafo, seed=42, iterations=15, k=0.2)
        except:
            pos = nx.random_layout(grafo, seed=42)
        
        nx.draw_networkx_edges(grafo, pos, alpha=0.08, width=0.2, ax=ax_grafo)
        nx.draw_networkx_nodes(grafo, pos, node_color=cores_lista,
                              node_size=50, cmap='Set3', vmin=0,
                              vmax=max(cores_set), ax=ax_grafo, linewidths=0)
        
        ax_grafo.set_title(f'Coloração de Vértices - {num_vertices} vértices\nχ(G) = {num_cores} cores',
                          fontsize=12, fontweight='bold')
        ax_grafo.axis('off')
        
        # Subplot 2: Distribuição de cores (barras horizontais + info)
        ax_stats = plt.subplot(1, 3, 3)
        cores_ids = sorted(distribuicao_cores.keys())
        contagens = [distribuicao_cores[c] for c in cores_ids]
        colors_bar = plt.cm.Set3(np.linspace(0, 1, num_cores))
        
        ax_stats.barh(cores_ids, contagens, color=colors_bar[:len(cores_ids)],
                     edgecolor='black', linewidth=1)
        ax_stats.set_ylabel('Cor', fontsize=10, fontweight='bold')
        ax_stats.set_xlabel('Vértices', fontsize=10, fontweight='bold')
        ax_stats.set_title('Distribuição', fontsize=11, fontweight='bold')
        ax_stats.grid(True, alpha=0.3, axis='x')
        ax_stats.invert_yaxis()
        
        info_text = f"Vértices: {num_vertices}\nArestas: {num_arestas}\nχ(G): {num_cores}\nDensidade: {densidade:.5f}\nGrau médio: {2*num_arestas/num_vertices:.2f}"
        ax_stats.text(0.02, 0.02, info_text, transform=ax_stats.transAxes,
                     fontsize=9, verticalalignment='bottom',
                     bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))
        
        dpi = 120
    
    plt.tight_layout()
    plt.savefig(caminho_saida, dpi=dpi, bbox_inches='tight')
    plt.close()

print("✓ Função visualizar_e_salvar_grafo definida")

✓ Função visualizar_e_salvar_grafo definida


## 4. Geração de Grafos Aleatórios

### Objetivo desta Seção (REQUISITO 2 - PARTE 1)
Implementar gerador de instâncias aleatórias de grafos usando o modelo Erdős-Rényi, que garante:
- Grafos não direcionados
- Sem ponderação
- Sem loops ou arestas paralelas

## Modelo Erdős-Rényi (ER)

### Características
- **Definição**: G(n, p) onde n = vértices e p = probabilidade de aresta
- **Funcionamento**: Para cada par (u,v), adiciona aresta com probabilidade p
- **Garantias**: Sem loops ou múltiplas arestas por design
- **Distribuição**: Graus seguem distribuição binomial

### Parâmetros da Função
- `num_vertices`: Tamanho do grafo (5 a 13 vértices para Parte 1)
- `probabilidade`: P(aresta existe) entre 0 e 1 (padrão 0.3)
- `seed`: Para reprodutibilidade de resultados

### Comportamento Especial
Se o grafo gerado estiver vazio (comum em grafos pequenos com p baixo), regenera até ter pelo menos uma aresta.


---

# 🔵 PARTE 1: FORÇA BRUTA COM INSTÂNCIAS ALEATÓRIAS

## Objetivo Geral
Implementar e analisar um **algoritmo exato** de coloração de grafos através de busca exaustiva (força bruta), avaliando sua escalabilidade em instâncias aleatórias pequenas (5-13 vértices).

## O que será feito:
1. **Gerar** grafos aleatórios (modelo Erdős-Rényi)
2. **Encontrar** número cromático exato χ(G) por força bruta
3. **Medir** tempo de execução e demonstrar crescimento exponencial
4. **Visualizar** grafos coloridos e gerar gráficos de análise

## Resultado esperado:
- 27 instâncias testadas (3 por tamanho, de 5-13 vértices)
- χ(G) exato para cada grafo
- Demonstração clara: por que força bruta falha para n > ~15

---



In [7]:
def gerar_instancia_grafo_aleatorio(num_vertices: int, probabilidade: float, 
                                   seed: int = None) -> nx.Graph:
    """
    REQUISITO 2 - PARTE 1: Gera instância aleatória de grafo.
    
    Cria um grafo aleatório usando modelo Erdős-Rényi:
    - Não direcionado
    - Não ponderado
    - Sem loops ou arestas paralelas (garantido pelo modelo)
    
    Args:
        num_vertices: Número de vértices (n)
        probabilidade: Probabilidade de aresta entre dois vértices (0 a 1)
        seed: Seed para reprodutibilidade
    
    Returns:
        Grafo NetworkX não direcionado
    """
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)
    
    # Modelo Erdős-Rényi: grafo aleatório
    grafo = nx.erdos_renyi_graph(n=num_vertices, p=probabilidade)
    
    # Garantir que o grafo tem arestas (para grafos muito pequenos)
    while grafo.number_of_edges() == 0 and num_vertices > 1:
        grafo = nx.erdos_renyi_graph(n=num_vertices, p=probabilidade)
    
    return grafo

print("✓ Função gerar_instancia_grafo_aleatorio definida")

✓ Função gerar_instancia_grafo_aleatorio definida


## 5. Algoritmo de Força Bruta para Coloração

### Objetivo desta Seção (REQUISITO 1 - PARTE 1)
Implementar algoritmo exato que encontra o número cromático χ(G) através de busca exaustiva de combinações de cores.

## Estratégia do Algoritmo

### Pseudocódigo
```
para k = 1 até n_vertices:
    para cada combinação de k cores em n_vertices:
        se coloracao é válida (vértices adjacentes diferem):
            retorna k, coloracao, tempo
```

### Busca por Cores Mínimas
1. Começa com k=1 cor
2. Se nenhuma combinação válida, tenta k=2
3. Continua até encontrar k válido
4. χ(G) é o primeiro k que funciona

## Validação de Coloração

Uma coloração é **válida** se:
- Cada vértice tem uma cor (0 a k-1)
- Para toda aresta (u,v): cor[u] ≠ cor[v]

## Complexidade e Limitações

### Complexidade Assintótica
- **Pior caso**: O(k^n) onde k=χ(G) e n=vértices
- **Típico para n≤13**: Viável (13^3 ≈ 2,197 combinações por k)
- **Para n>15**: Tempo exponencial proibitivo

### Razão de Ser
- Encontra solução **ótima** (χ(G) exato)
- Maior tamanho testável: ~15-20 vértices com hardware moderno

## Saída Esperada
Tupla contendo:
- `numero_cromatico`: Valor exato de χ(G)
- `coloracao`: Dicionário {vértice: cor}
- `tempo_segundos`: Tempo de execução medido

In [8]:
def algoritmo_forca_bruta_coloracao(grafo: nx.Graph) -> Tuple[int, Dict, float]:
    """
    REQUISITO 1 - PARTE 1: Implementa algoritmo de força bruta para coloração.
    
    ESTRATÉGIA OTIMIZADA COM BACKTRACKING E PODA:
    -----------------------------------------------
    Para cada número de cores k = 1, 2, 3, ...:
      1. Usa backtracking para colorir vértices um por um
      2. Descarta (poda) caminhos inválidos imediatamente
      3. Se encontrar coloração válida, retorna k como número cromático
    
    OTIMIZAÇÕES:
    - Backtracking: Para imediatamente ao detectar conflito
    - Poda: Não explora ramos inválidos (reduz k^n drasticamente)
    - Ordenação: Processa vértices por grau decrescente (melhor poda)
    
    Complexidade: O(k^n) pior caso, mas com poda eficiente na prática
    Viável para n ≤ ~15-20 vértices
    
    Args:
        grafo: Grafo NetworkX
        
    Returns:
        Tupla: (número_cromático, coloração_dict, tempo_segundos)
    """
    inicio = time.time()
    
    vertices = list(grafo.nodes())
    n_vertices = len(vertices)
    
    # Ordenar vértices por grau decrescente para melhor poda
    vertices_ordenados = sorted(vertices, key=lambda v: grafo.degree(v), reverse=True)
    
    # Pré-calcular vizinhos para acesso rápido
    vizinhos = {v: set(grafo.neighbors(v)) for v in vertices}
    
    def backtrack(indice: int, coloracao: Dict, num_cores: int) -> bool:
        """
        Tenta colorir vértices a partir do índice usando backtracking.
        
        Args:
            indice: Índice do vértice atual
            coloracao: Dicionário de coloração atual
            num_cores: Número máximo de cores disponíveis
            
        Returns:
            True se encontrou coloração válida, False caso contrário
        """
        # Caso base: todos os vértices foram coloridos
        if indice == n_vertices:
            return True
        
        vertice_atual = vertices_ordenados[indice]
        
        # Determinar cores já usadas por vizinhos
        cores_vizinhos = set()
        for vizinho in vizinhos[vertice_atual]:
            if vizinho in coloracao:
                cores_vizinhos.add(coloracao[vizinho])
        
        # Tentar cada cor disponível
        for cor in range(num_cores):
            # PODA: Se cor já usada por vizinho, pular
            if cor in cores_vizinhos:
                continue
            
            # Atribuir cor ao vértice atual
            coloracao[vertice_atual] = cor
            
            # Recursivamente colorir próximo vértice
            if backtrack(indice + 1, coloracao, num_cores):
                return True
            
            # Backtrack: remover coloração
            del coloracao[vertice_atual]
        
        # Nenhuma cor funcionou
        return False
    
    # Tentar com 1, 2, 3, ... cores até encontrar coloração válida
    for num_cores in range(1, n_vertices + 1):
        coloracao = {}
        if backtrack(0, coloracao, num_cores):
            tempo = time.time() - inicio
            return num_cores, coloracao, tempo
    
    # Nunca deve chegar aqui (coloração sempre existe)
    tempo = time.time() - inicio
    return n_vertices, {v: i for i, v in enumerate(vertices)}, tempo

print("✓ Função algoritmo_forca_bruta_coloracao definida (OTIMIZADA com backtracking e poda)")

✓ Função algoritmo_forca_bruta_coloracao definida (OTIMIZADA com backtracking e poda)


## 6. Processamento de Instâncias com Força Bruta

### Objetivo desta Seção (REQUISITO 2 + 3 - PARTE 1)
Gerar múltiplas instâncias aleatórias de tamanhos variados, aplicar força bruta a cada uma e medir tempo de execução para análise de escalabilidade.

## Fluxo de Processamento

### Para cada tamanho de 5 a 13 vértices:
1. Gera 3 instâncias aleatórias diferentes (usando seeds diferentes)
2. Extrai parâmetros estatísticos de cada grafo
3. Aplica algoritmo de força bruta
4. Registra tempo de execução
5. Visualiza grafo colorido como PNG

## Matriz de Testes

| Tamanho | Instâncias | Total de Testes |
|---------|-----------|-----------------|
| 5-13    | 3 cada    | 27 grafos       |

**Total esperado**: 27 CSVs + 27 PNGs + 2 gráficos de análise

## Dados Coletados por Instância

### Parâmetros do Grafo
- Número de vértices, arestas
- Densidade, grau médio, grau mín/máx
- Conexidade, número de componentes

### Resultados de Coloração
- ID único (n{tamanho:02d}_i{instância:02d})
- Número cromático encontrado χ(G)
- **Tempo de execução (segundos)** - crítico para análise
- Número de arestas

## Exportação

### CSV de Parâmetros
Armazena características de cada grafo para análise posterior (densidade vs cores, etc.)

### CSV de Resultados
Armazena χ(G) e tempo por instância - essencial para gráfico de escalabilidade

### PNG de Grafos
Visualizações coloridas de cada grafo processado

In [9]:
def processar_instancias_por_tamanho(tamanho_min: int = 5, tamanho_max: int = 13,
                                     probabilidade: float = 0.3,
                                     num_instancias_por_tamanho: int = 3) -> Tuple[List, List]:
    """
    REQUISITO 2 + 3 - PARTE 1: Processa múltiplas instâncias aleatórias.
    
    Gera instâncias de tamanho tamanho_min a tamanho_max e aplica força bruta em cada.
    Mede tempo de execução para cada instância.
    
    Args:
        tamanho_min: Tamanho mínimo (padrão 5 vértices)
        tamanho_max: Tamanho máximo (padrão 13 vértices)
        probabilidade: Probabilidade de aresta entre vértices
        num_instancias_por_tamanho: Número de instâncias por tamanho
        
    Returns:
        Tupla: (lista_parametros, lista_resultados)
    """
    lista_parametros = []
    lista_resultados = []
    
    print("\n" + "="*70)
    print("PARTE 1: PROCESSANDO INSTÂNCIAS ALEATÓRIAS COM FORÇA BRUTA")
    print("="*70)
    
    for tamanho in range(tamanho_min, tamanho_max + 1):
        print(f"\nTamanho: {tamanho} vértices", end="")
        
        for instancia_idx in range(num_instancias_por_tamanho):
            # Gerar instância
            seed = tamanho * 1000 + instancia_idx
            grafo = gerar_instancia_grafo_aleatorio(tamanho, probabilidade, seed=seed)
            
            # Extrair parâmetros
            parametros = extrair_parametros_grafo(grafo)
            parametros['tamanho'] = tamanho
            parametros['instancia'] = instancia_idx + 1
            parametros['id_grafo'] = f"n{tamanho:02d}_i{instancia_idx+1:02d}"
            
            # Aplicar força bruta
            try:
                num_cromatico, coloracao, tempo = algoritmo_forca_bruta_coloracao(grafo)
                
                resultado = {
                    'id_grafo': parametros['id_grafo'],
                    'tamanho': tamanho,
                    'instancia': instancia_idx + 1,
                    'numero_cromatico': num_cromatico,
                    'tempo_segundos': tempo,
                    'arestas': parametros['num_arestas'],
                }
                
                lista_parametros.append(parametros)
                lista_resultados.append(resultado)
                
                # Salvar visualização
                caminho_grafo = ConfiguradorDiretorios.obter_caminho_arquivo(
                    'parte1', 'grafos', f"grafo_{parametros['id_grafo']}.png"
                )
                visualizar_e_salvar_grafo(grafo, coloracao, caminho_grafo)
                
                print(".", end="", flush=True)
            
            except KeyboardInterrupt:
                print("\n⚠ Execução interrompida pelo usuário")
                break
            except Exception as e:
                print(f"\n⚠ Erro ao processar grafo {tamanho}_{instancia_idx}: {str(e)}")
                continue
        
        print(f" [{tamanho} vértices processado]")
    
    print("\n✓ Processamento de instâncias concluído")
    return lista_parametros, lista_resultados

print("✓ Função processar_instancias_por_tamanho definida")

✓ Função processar_instancias_por_tamanho definida


## 7. Algoritmo Heurístico Welsh-Powell

### Objetivo desta Seção (REQUISITO 1 - PARTE 2)
Implementar heurística gulosa de coloração que encontra solução **viável** (válida mas não necessariamente ótima) em tempo linear.

## Estratégia Welsh-Powell

### Pseudocódigo
```
1. Ordena vértices por grau DECRESCENTE
2. Para cada vértice em ordem:
   a. Encontra cores usadas pelos vizinhos já coloridos
   b. Atribui a MENOR cor não-usada
3. Retorna número de cores usadas
```

### Exemplo Ilustrativo
```
Grafo com graus: A(3), B(2), C(2), D(1), E(1)
Ordem: A → B → C → D → E

Processamento:
- A: cor 0 (primeira disponível)
- B: vizinho A(0), usa cor 1
- C: vizinho A(0), usa cor 1
- D: vizinho B(1), C(1), usa cor 0
- E: vizinho A(0), B(1), usa cor 2

Resultado: 3 cores
```

## Propriedades da Heurística

### Complexidade
- **Tempo**: O(n + m) - Linear em vértices e arestas
- **Espaço**: O(n) para armazenar coloração

### Garantias de Qualidade
- **Válida**: Sempre produz coloração válida
- **Limitante Superior**: Usa no máximo Δ+1 cores, onde Δ=grau máximo
- **Não-Ótima**: Pode usar mais cores que χ(G) ótimo

### Efetividade
- Funciona bem para grafos com distribuição de graus não uniforme
- Especialmente bom para grafos esparsos
- Ordem de processamento é crítica para qualidade

## Saída Esperada
Tupla contendo:
- `numero_cores`: Cores usadas pela heurística
- `coloracao`: Dicionário {vértice: cor}
- `tempo_segundos`: Tempo muito pequeno (< 0.01s típico)


---

# 🔴 PARTE 2: HEURÍSTICA EM INSTÂNCIAS DIMACS REAIS

## Objetivo Geral
Implementar e aplicar um **algoritmo heurístico** (Welsh-Powell) em instâncias grandes do mundo real, demonstrando eficiência prática quando força bruta é inviável.

## O que será feito:
1. **Ler** 5 arquivos DIMACS (instâncias reais do Moodle)
2. **Aplicar** heurística Welsh-Powell (tempo linear O(n+m))
3. **Encontrar** χ_heurística (solução viável, não necessariamente ótima)
4. **Comparar** com força bruta: escalabilidade vs otimalidade

## Resultado esperado:
- 5 instâncias processadas (450-4730 vértices)
- Cores encontradas pela heurística
- Demonstração: heurísticas são essenciais para problemas grandes
- Análise: relação entre densidade, tamanho e qualidade da solução

---



In [10]:
def algoritmo_welsh_powell(grafo: nx.Graph) -> Tuple[int, Dict, float]:
    """
    REQUISITO 1 - PARTE 2: Implementa heurística Welsh-Powell para coloração.
    
    ESTRATÉGIA:
    -----------
    Algoritmo guloso que ordena vértices por grau decrescente:
    1. Ordena vértices por grau (maior primeiro)
    2. Para cada vértice em ordem:
       - Atribui a menor cor disponível (que não foi usada por vizinhos)
    
    Complexidade: O(n + m) - Linear em número de vértices e arestas
    Garantias: Encontra coloração válida (nem sempre ótima)
    Qualidade: Típicamente O(Δ + 1) cores, onde Δ é grau máximo
    
    Args:
        grafo: Grafo NetworkX
        
    Returns:
        Tupla: (número_cores_usado, coloracao_dict, tempo_segundos)
    """
    inicio = time.time()
    
    # Ordenar vértices por grau decrescente (heurística Welsh-Powell)
    graus = dict(grafo.degree())
    vertices_ordenados = sorted(graus.keys(), key=lambda v: graus[v], reverse=True)
    
    coloracao = {}
    
    # Para cada vértice em ordem de grau decrescente
    for vertice in vertices_ordenados:
        # Cores já usadas pelos vizinhos
        cores_vizinhos = set()
        for vizinho in grafo.neighbors(vertice):
            if vizinho in coloracao:
                cores_vizinhos.add(coloracao[vizinho])
        
        # Encontrar menor cor disponível
        cor = 0
        while cor in cores_vizinhos:
            cor += 1
        
        coloracao[vertice] = cor
    
    tempo = time.time() - inicio
    num_cores = max(coloracao.values()) + 1 if coloracao else 0
    
    return num_cores, coloracao, tempo

print("✓ Função algoritmo_welsh_powell definida")

✓ Função algoritmo_welsh_powell definida


## 8. Algoritmo Heurístico DSATUR (Adicional)

### Objetivo desta Seção
Implementar heurística **DSATUR** (Degree of Saturation) para coloração, uma técnica mais sofisticada que Welsh-Powell, com melhor qualidade de solução.

## Estratégia DSATUR (Degree of Saturation)

### Conceito Principal
DSATUR usa uma **heurística adaptativa** que escolhe vértices baseado na **saturação** (número de cores diferentes já usadas pelos vizinhos).

### Pseudocódigo
```
1. Colore vértice de maior grau com cor 0
2. Enquanto houver vértices não coloridos:
   a. Calcula DSAT de cada vértice não colorido
      DSAT = número de cores diferentes dos vizinhos coloridos
   b. Escolhe vértice com MAIOR DSAT (desempate: maior grau)
   c. Atribui menor cor disponível
3. Retorna número de cores usadas
```

### Exemplo Ilustrativo
```
Grafo: A--B--C--D
       |     |
Graus: A(2), B(2), C(2), D(1)

Passo 1: A recebe cor 0 (maior grau)
Passo 2: B tem DSAT=1 (vizinho A cor 0), recebe cor 1
Passo 3: C tem DSAT=1 (vizinho B cor 1), recebe cor 0
Passo 4: D tem DSAT=2 (vizinhos C e C), recebe cor 2

Resultado: 3 cores (pode ser melhor que Welsh-Powell)
```

## Propriedades da Heurística

### Complexidade
- **Tempo**: O(n²) - Quadrática (mais lenta que Welsh-Powell)
- **Espaço**: O(n) para estruturas auxiliares

### Garantias de Qualidade
- **Válida**: Sempre produz coloração válida
- **Qualidade Superior**: Geralmente 5-10% menos cores que Welsh-Powell
- **Adaptativa**: Considera estado atual da coloração (não apenas grau estático)


### Quando usar DSATUR?- Problema de coloração real onde cores extras custam caro

- Grafos densos onde qualidade é crucial- Benchmarking contra outras heurísticas
- Quando o tempo extra O(n²) é aceitável

In [11]:
def algoritmo_dsatur(grafo: nx.Graph) -> Tuple[int, Dict, float]:
    """
    Implementa heurística DSATUR para coloração de vértices.
    
    ESTRATÉGIA DSATUR (Degree of SATuration):
    ------------------------------------------
    Algoritmo mais sofisticado que considera saturação de vértices:
    1. Começa com vértice de maior grau
    2. Em cada passo, escolhe vértice NÃO colorido com MAIOR DSAT:
       - DSAT = número de cores diferentes dos vizinhos já coloridos
    3. Atribui menor cor disponível
    
    Complexidade: O(n²) - Quadrática, mas ainda prática
    Garantias: Encontra coloração válida
    Qualidade: Típicamente melhor que Welsh-Powell (~5-10% menos cores)
    
    Args:
        grafo: Grafo NetworkX
        
    Returns:
        Tupla: (número_cores_usado, coloracao_dict, tempo_segundos)
    """
    inicio = time.time()
    
    coloracao = {}
    
    # Encontrar vértice com maior grau para começar
    if not grafo.nodes():
        return 0, {}, 0
    
    graus = dict(grafo.degree())
    primeiro = max(graus.keys(), key=lambda v: graus[v])
    
    coloracao[primeiro] = 0
    
    # Processar vértices restantes
    while len(coloracao) < len(grafo.nodes()):
        # Encontrar vértice não colorido com maior DSAT
        melhor_vertice = None
        max_dsat = -1
        
        for v in grafo.nodes():
            if v not in coloracao:
                # Calcular DSAT: número de cores diferentes dos vizinhos coloridos
                cores_vizinhos = set()
                for vizinho in grafo.neighbors(v):
                    if vizinho in coloracao:
                        cores_vizinhos.add(coloracao[vizinho])
                
                dsat = len(cores_vizinhos)
                
                # Desempate: usar grau como critério secundário
                if dsat > max_dsat or (dsat == max_dsat and (melhor_vertice is None or graus[v] > graus[melhor_vertice])):
                    melhor_vertice = v
                    max_dsat = dsat
        
        if melhor_vertice is not None:
            # Encontrar menor cor disponível
            cores_vizinhos = set()
            for vizinho in grafo.neighbors(melhor_vertice):
                if vizinho in coloracao:
                    cores_vizinhos.add(coloracao[vizinho])
            
            cor = 0
            while cor in cores_vizinhos:
                cor += 1
            
            coloracao[melhor_vertice] = cor
    
    tempo = time.time() - inicio
    num_cores = max(coloracao.values()) + 1 if coloracao else 0
    
    return num_cores, coloracao, tempo

print("✓ Função algoritmo_dsatur definida")

✓ Função algoritmo_dsatur definida


**Nota**: A função `ler_arquivo_dimacs()` já foi definida nas funções auxiliares comuns (seção 3). Ela é compartilhada por todos os algoritmos que processam instâncias DIMACS.

## 9. Processamento de Instâncias DIMACS (DUAS Heurísticas)

### Objetivo desta Seção (REQUISITO 2 + 3 - PARTE 2)
Processar 5 instâncias reais do problema de coloração do Moodle, aplicar **DUAS heurísticas** (Welsh-Powell e DSATUR) e registrar resultados separados para comparação.

## Instâncias do Moodle

| ID | Nome | Vértices | Arestas | Referência |
|----|------|----------|---------|-----------|
| a  | le450_25a | 450 | 8,260 | DIMACS Competitive Challenge |
| b  | inithx.i.1 | 864 | 18,707 | Benchmark Real-World |
| c  | r1000.1 | 1,000 | 14,378 | Random Generated |
| d  | ash958GPIA | 1,916 | 12,506 | Geometric Structure |
| e  | wap03a | 4,730 | 286,722 | **Grande - Teste de Escalabilidade** |

## Características das Instâncias

### Tamanho e Complexidade
- Instâncias **progressivamente maiores**
- Instância 'e' tem ~60x mais arestas que 'a'
- Força bruta **não é viável** para estas (n > 20)

### Tipos de Estrutura
- **Random** (r1000.1): Distribuição uniforme
- **Geométrica** (ash958GPIA): Baseada em coordenadas
- **Real-World** (inithx.i.1, wap03a): Problemas práticos
- **DIMACS** (le450_25a): Benchmark clássico

## Processamento por Instância

### Para cada arquivo DIMACS:
1. Lê arquivo usando `ler_arquivo_dimacs()`
2. Aplica `algoritmo_welsh_powell()` → salva em `heuristica_Welsh-Powell.csv`
3. Aplica `algoritmo_dsatur()` → salva em `heuristica_DSATUR.csv`
4. Registra para **CADA** heurística:
   - Número de cores encontrado
   - Tempo de execução
   - Densidade do grafo
   - Grau médio
   - Parâmetros do grafo
5. Visualiza ambas colorações (limite: 500 vértices)
6. Cria `comparacao_algoritmos.csv` com os 3 algoritmos

## Dados Coletados (CSVs Separados)
- **Welsh-Powell**: `heuristica_Welsh-Powell.csv`
- **DSATUR**: `heuristica_DSATUR.csv`
- **Informações**: `informacoes_instancias.csv` (métricas consolidadas)
- **Comparação**: `comparacao_algoritmos.csv` (Força Bruta vs WP vs DSATUR)

In [12]:
def processar_instancias_dimacs() -> Dict:
    """
    REQUISITO 2 + 3 - PARTE 2: Processa instâncias DIMACS com DUAS heurísticas.
    
    Aplica ambos os algoritmos nas 5 instâncias:
    1. Welsh-Powell (greedy por grau decrescente)
    2. DSATUR (greedy por saturação adaptativa)
    
    Instâncias esperadas (do Moodle):
      a. 450 vértices, 8260 arestas
      b. 864 vértices, 18707 arestas
      c. 1000 vértices, 14378 arestas
      d. 1916 vértices, 12506 arestas
      e. 4730 vértices, 286722 arestas
    
    Returns:
        Dicionário com resultados separados:
        {
            'resultados_welsh_powell': [...],
            'resultados_dsatur': [...],
            'informacoes': [...]
        }
    """
    resultados_welsh_powell = []
    resultados_dsatur = []
    informacoes = []
    
    # Definir instâncias esperadas
    instancias = [
        {
            'id': 'a',
            'caminho': 'instancias/a - le450_25a.col.txt',
            'vertices_esperados': 450,
            'arestas_esperadas': 8260,
        },
        {
            'id': 'b',
            'caminho': 'instancias/b - inithx.i.1.col.txt',
            'vertices_esperados': 864,
            'arestas_esperadas': 18707,
        },
        {
            'id': 'c',
            'caminho': 'instancias/c - r1000.1.col.txt',
            'vertices_esperados': 1000,
            'arestas_esperadas': 14378,
        },
        {
            'id': 'd',
            'caminho': 'instancias/d - ash958GPIA.col.txt',
            'vertices_esperados': 1916,
            'arestas_esperadas': 12506,
        },
        {
            'id': 'e',
            'caminho': 'instancias/e - wap03a.col.txt',
            'vertices_esperados': 4730,
            'arestas_esperadas': 286722,
        },
    ]
    
    print("\n" + "="*80)
    print("PARTE 2: PROCESSANDO INSTÂNCIAS DIMACS COM DUAS HEURÍSTICAS")
    print("="*80)
    
    for instancia in instancias:
        print(f"\n📄 Instância {instancia['id']} ({instancia['caminho']})")
        
        grafo = ler_arquivo_dimacs(instancia['caminho'])
        
        if grafo is None:
            print(f"  ⚠️  Arquivo não encontrado")
            continue
        
        if grafo.number_of_nodes() == 0:
            print(f"  ⚠️  Grafo vazio")
            continue
        
        n = grafo.number_of_nodes()
        m = grafo.number_of_edges()
        
        print(f"  ✓ Carregado: {n} vértices, {m} arestas")
        
        # WELSH-POWELL
        try:
            print(f"    🔹 Executando Welsh-Powell... ", end="", flush=True)
            num_cores_wp, coloracao_wp, tempo_wp = algoritmo_welsh_powell(grafo)
            
            resultados_welsh_powell.append({
                'instancia_id': instancia['id'],
                'caminho': instancia['caminho'],
                'num_vertices': n,
                'num_arestas': m,
                'cores_heuristica': num_cores_wp,
                'tempo_segundos': tempo_wp,
                'densidade': nx.density(grafo),
                'grau_medio': sum(dict(grafo.degree()).values()) / n,
            })
            
            print(f"✓ {num_cores_wp} cores em {tempo_wp:.4f}s")
            
            # Salvar visualização Welsh-Powell
            caminho_grafo_wp = ConfiguradorDiretorios.obter_caminho_arquivo(
                'parte2', 'grafos', f"instancia_{instancia['id']}_welsh_powell.png"
            )
            visualizar_e_salvar_grafo(grafo, coloracao_wp, caminho_grafo_wp)
            
        except Exception as e:
            print(f"⚠️ Erro: {str(e)}")
        
        # DSATUR
        try:
            print(f"    🔸 Executando DSATUR... ", end="", flush=True)
            num_cores_ds, coloracao_ds, tempo_ds = algoritmo_dsatur(grafo)
            
            resultados_dsatur.append({
                'instancia_id': instancia['id'],
                'caminho': instancia['caminho'],
                'num_vertices': n,
                'num_arestas': m,
                'cores_heuristica': num_cores_ds,
                'tempo_segundos': tempo_ds,
                'densidade': nx.density(grafo),
                'grau_medio': sum(dict(grafo.degree()).values()) / n,
            })
            
            print(f"✓ {num_cores_ds} cores em {tempo_ds:.4f}s")
            
            # Salvar visualização DSATUR
            caminho_grafo_ds = ConfiguradorDiretorios.obter_caminho_arquivo(
                'parte2', 'grafos', f"instancia_{instancia['id']}_dsatur.png"
            )
            visualizar_e_salvar_grafo(grafo, coloracao_ds, caminho_grafo_ds)
            
        except Exception as e:
            print(f"⚠️ Erro: {str(e)}")
        
        # Salvar informações consolidadas
        informacoes.append({
            'instancia': instancia['id'],
            'num_vertices': n,
            'num_arestas': m,
            'densidade': round(nx.density(grafo), 4),
            'grau_medio': round(sum(dict(grafo.degree()).values()) / n, 2),
            'cores_welsh_powell': num_cores_wp if 'num_cores_wp' in locals() else 'N/A',
            'cores_dsatur': num_cores_ds if 'num_cores_ds' in locals() else 'N/A',
        })
    
    print("\n✓ Processamento de instâncias DIMACS concluído")
    print(f"  • Welsh-Powell: {len(resultados_welsh_powell)} instâncias processadas")
    print(f"  • DSATUR: {len(resultados_dsatur)} instâncias processadas")
    
    return {
        'resultados_welsh_powell': resultados_welsh_powell,
        'resultados_dsatur': resultados_dsatur,
        'informacoes': informacoes
    }

print("✓ Função processar_instancias_dimacs definida (com Welsh-Powell E DSATUR)")


✓ Função processar_instancias_dimacs definida (com Welsh-Powell E DSATUR)


In [13]:
# Criar estrutura de diretórios usando ConfiguradorDiretorios
# Isso cria a estrutura completa com parte1 e parte2
ConfiguradorDiretorios.criar_estrutura()
print("✓ Estrutura de diretórios criada com sucesso")

✓ Estrutura de diretórios criada com sucesso
✓ Estrutura de diretórios criada com sucesso


## 10. Teste de Estrutura de Diretórios

### Objetivo desta Seção
Verificar que toda a estrutura de diretórios necessária foi criada corretamente antes de executar a solução completa.

### O que faz:
- Cria todos os diretórios em `resultados_1/`
- Garante que CSVs, grafos e gráficos podem ser salvos
- Evita erros de "diretório não encontrado" durante execução

### Estrutura Completa:
```
resultados_1/
├── parte1/
│   ├── csv/       ✓ Parâmetros e resultados força bruta
│   ├── grafos/    ✓ 33 grafos coloridos (PNG)
│   └── graficos/  ✓ Gráficos de escalabilidade
└── parte2/
    ├── csv/       ✓ 6 CSVs (2 heurísticas + comparação)
    ├── grafos/    ✓ 10 grafos coloridos (5 WP + 5 DSATUR)
    └── graficos/  ✓ Gráficos comparativos
```

In [14]:
ConfiguradorDiretorios.criar_estrutura()
print("✓ Estrutura de diretórios criada com sucesso")

✓ Estrutura de diretórios criada com sucesso
✓ Estrutura de diretórios criada com sucesso


## 10. Exportação de Resultados para CSV

### Objetivo desta Seção
Exportar todos os resultados em formato CSV para análise posterior, relatórios e criação de gráficos.

## Formato de Saída

### Parte 1: `parametros_grafos.csv`
Armazena características de cada instância gerada:
```
num_vertices, num_arestas, densidade, grau_medio, grau_minimo, grau_maximo, 
eh_conexo, num_componentes, tamanho, instancia, id_grafo
```

**Uso**: Análise de correlação entre propriedades do grafo e número cromático

### Parte 1: `resultados_forca_bruta.csv`
Armazena resultados da força bruta:
```
id_grafo, tamanho, instancia, numero_cromatico, tempo_segundos, arestas
```

**Uso**: Gráfico de escalabilidade (tempo vs tamanho)

### Parte 2: `resultados_heuristica.csv`
Armazena resultados da heurística Welsh-Powell:
```
instancia_id, caminho, num_vertices, num_arestas, cores_heuristica, 
tempo_segundos, densidade, grau_medio
```

**Uso**: 
- Comparação com força bruta (qualidade da heurística)
- Análise de performance em instâncias grandes
- Correlação entre propriedades e qualidade

## Vantagens do CSV
- ✓ Formato universal (Excel, R, Python, etc.)
- ✓ Fácil de processar com pandas
- ✓ Legível como texto
- ✓ Suporta análises estatísticas posteriores
- ✓ Documentação automática de resultados

## 11. Geração de Gráficos de Análise - Parte 1

### Objetivo desta Seção
Criar visualizações que demonstram o crescimento exponencial da força bruta e analisam o número cromático encontrado.

## Gráfico 1: Escalabilidade (Tempo vs Tamanho)

### Características
- **Eixo Y (log)**: Tempo em escala logarítmica (semilogy)
  - Revelam crescimento exponencial como reta
- **Eixo X (linear)**: Tamanho do grafo em vértices
- **Linha azul**: Tempo **médio** por tamanho
- **Área sombreada**: Intervalo min-max observado

### Interpretação
- **Inclinação suave**: Crescimento polinomial
- **Inclinação íngreme**: Crescimento exponencial
- **Espaçamento**: Variabilidade entre instâncias

### Propósito
Demonstra por que força bruta é inviável para n > ~15:
- n=10: ~0.001s
- n=13: ~1-10s
- n=15: ~1000s+ (proibitivo)

---

## Gráfico 2: Número Cromático vs Tamanho

### Características
- **Linha verde** com marcadores quadrados
- Mostra evolução de χ(G) com tamanho
- Linha de tendência implícita pela observação

### Observações Esperadas
- χ(G) não cresce linearmente com n
- Influência forte da densidade
- Máximo teórico: χ ≤ n

### Propósito
Validar que algoritmo encontra diferentes valores cromáticos
- Não é constante (grafo muda)
- Não cresce trivialmente

In [15]:
def gerar_instancias_aleatorias(tamanho_min: int = 5, tamanho_max: int = 15, 
                                instancias_por_tamanho: int = 3) -> List:
    """
    Wrapper que gera instâncias aleatórias de grafos.
    
    REQUISITO: O trabalho especifica "tamanho máximo vai girar em torno de 13 a 20"
    Intervalo padrão: 5-15 vértices (33 instâncias, 3 por tamanho)
    
    Args:
        tamanho_min: Tamanho mínimo (padrão: 5 vértices)
        tamanho_max: Tamanho máximo (padrão: 15 vértices - pode tentar até 20)
        instancias_por_tamanho: Quantas instâncias por tamanho (padrão: 3)
    
    Retorna:
        Lista de dicionários com parâmetros e grafos gerados
    """
    instancias = []
    
    print(f"\n📊 Gerando instâncias aleatórias: {tamanho_min}-{tamanho_max} vértices")
    print(f"   Total esperado: {(tamanho_max - tamanho_min + 1) * instancias_por_tamanho} instâncias\n")
    
    for tamanho in range(tamanho_min, tamanho_max + 1):
        for i in range(instancias_por_tamanho):
            seed = tamanho * 100 + i
            # Probabilidade aumenta com tamanho (grafos maiores = mais densos)
            prob = 0.3 + (tamanho - tamanho_min) * 0.05
            
            grafo = gerar_instancia_grafo_aleatorio(
                num_vertices=tamanho,
                probabilidade=min(prob, 0.9),
                seed=seed
            )
            
            instancias.append({
                'tamanho': tamanho,
                'grafo': grafo,
                'probabilidade': prob,
                'seed': seed
            })
    
    return instancias


def processar_instancias_aleatorias(instancias: List) -> List:
    """
    Processa instâncias aleatórias usando força bruta.
    
    IMPORTANTE: Força bruta tem complexidade exponencial!
    - Grafos até 13 vértices: Segundos
    - Grafos 14-15 vértices: Minutos
    - Grafos 16-20 vértices: Horas (pode ser inviável)
    
    Args:
        instancias: Lista de dicionários com grafos
        
    Returns:
        Lista de resultados (parâmetros e números cromáticos)
    """
    parametros_list = []
    resultados_list = []
    
    total = len(instancias)
    print(f"\n🔄 Processando {total} instâncias com Força Bruta...")
    print("⚠️  ATENÇÃO: Grafos maiores podem levar MUITO tempo!\n")
    
    for i, instancia in enumerate(instancias, 1):
        grafo = instancia['grafo']
        tamanho = instancia['tamanho']
        
        # Calcular parâmetros estruturais
        parametros = {
            'id_instancia': i,
            'tamanho': tamanho,
            'num_vertices': grafo.number_of_nodes(),
            'num_arestas': grafo.number_of_edges(),
            'densidade': nx.density(grafo),
            'grau_medio': sum(dict(grafo.degree()).values()) / grafo.number_of_nodes() if grafo.number_of_nodes() > 0 else 0
        }
        parametros_list.append(parametros)
        
        # Executar força bruta
        try:
            print(f"  [{i}/{total}] Tamanho {tamanho}: ", end="", flush=True)
            inicio_inst = time.time()
            
            numero_cromatico, coloracao, tempo = algoritmo_forca_bruta_coloracao(grafo)
            
            resultado = {
                'id_instancia': i,
                'tamanho': tamanho,
                'num_vertices': grafo.number_of_nodes(),
                'num_arestas': grafo.number_of_edges(),
                'num_cores': numero_cromatico,
                'tempo_execucao': tempo,
                'densidade': nx.density(grafo),
                'grafo': grafo,
                'coloracao': coloracao
            }
            resultados_list.append(resultado)
            
            print(f"χ(G)={numero_cromatico}, tempo={tempo:.4f}s ✓")
            
            # Salvar visualização do grafo
            caminho_grafo = ConfiguradorDiretorios.obter_caminho_arquivo(
                'parte1', 'grafos', f"grafo_inst{i:02d}_n{tamanho:02d}.png"
            )
            visualizar_e_salvar_grafo(grafo, coloracao, caminho_grafo)
                
        except Exception as e:
            print(f"⚠️ ERRO: {str(e)}")
    
    # Combinar parâmetros e resultados
    return parametros_list + resultados_list

print("✓ Funções gerar_instancias_aleatorias e processar_instancias_aleatorias definidas")
print("  Intervalo padrão: 5-15 vértices (pode ajustar até 20 se tempo permitir)")

✓ Funções gerar_instancias_aleatorias e processar_instancias_aleatorias definidas
  Intervalo padrão: 5-15 vértices (pode ajustar até 20 se tempo permitir)


In [16]:
def consolidar_e_salvar_dados_csv():
    """
    Consolida e valida todos os CSVs gerados no fluxo de execução.
    Verifica a origem e integridade dos dados antes de usar em gerar_tabelas_latex.
    
    Inclui as 3 análises principais:
    1. Força Bruta (instâncias aleatórias)
    2. Heurística Welsh-Powell (instâncias DIMACS)
    3. Heurística DSATUR (instâncias DIMACS)
    
    Retorna:
        dict: Dicionário com informações sobre os CSVs encontrados e validados
    """
    
    print("\n" + "="*70)
    print("CONSOLIDANDO DADOS E SALVANDO EM CSV")
    print("="*70)
    
    print("\n📊 ORIGEM DOS DADOS (CSVs gerados automaticamente):\n")
    
    print("PARTE 1 - FORÇA BRUTA (Instâncias Aleatórias):")
    print("  1️⃣  parametros_grafos.csv")
    print("      Origem: processar_instancias_aleatorias()")
    print("      Conteúdo: Parâmetros estruturais dos grafos aleatórios gerados")
    print("      (tamanho: 5-15 vértices padrão, 3 instâncias cada)")
    print()
    print("  2️⃣  resultados_forca_bruta.csv")
    print("      Origem: algoritmo_forca_bruta_coloracao()")
    print("      Conteúdo: Número cromático encontrado e tempo para cada grafo")
    print()
    
    print("PARTE 2A - HEURÍSTICA WELSH-POWELL (Instâncias DIMACS):")
    print("  3️⃣  heuristica_Welsh-Powell.csv")
    print("      Origem: processar_instancias_dimacs() → algoritmo_welsh_powell()")
    print("      Conteúdo: Cores encontradas, tempo e parâmetros das 5 instâncias DIMACS")
    print()
    
    print("PARTE 2B - HEURÍSTICA DSATUR (Instâncias DIMACS):")
    print("  4️⃣  heuristica_DSATUR.csv")
    print("      Origem: processar_instancias_dimacs() → algoritmo_dsatur()")
    print("      Conteúdo: Cores encontradas, tempo e parâmetros das 5 instâncias DIMACS")
    print()
    
    print("COMPARAÇÃO CONSOLIDADA:")
    print("  5️⃣  comparacao_algoritmos.csv")
    print("      Origem: executar_solucao_completa()")
    print("      Conteúdo: Resumo comparativo entre Força Bruta, Welsh-Powell e DSATUR")
    print()
    
    print("INFORMAÇÕES CONSOLIDADAS:")
    print("  6️⃣  informacoes_instancias.csv")
    print("      Origem: processar_instancias_dimacs()")
    print("      Conteúdo: Informações consolidadas das instâncias DIMACS")
    print("      (densidade, grau médio, cores de ambas heurísticas, tempos)")
    print()
    
    # Verificar quais arquivos CSV foram gerados
    import os
    csv_files = []
    csv_info = {}
    
    print("-" * 70)
    print("Verificando arquivos CSV gerados em resultados_1...\n")
    
    for root, dirs, files in os.walk('resultados_1'):
        for file in sorted(files):
            if file.endswith('.csv'):
                path = os.path.join(root, file)
                csv_files.append(path)
                try:
                    df = pd.read_csv(path)
                    csv_info[file] = {
                        'path': path,
                        'linhas': len(df),
                        'colunas': len(df.columns),
                        'dataframe': df,
                        'status': '✓'
                    }
                    print(f"✓ {path}")
                    print(f"    Linhas: {len(df)} | Colunas: {len(df.columns)}")
                    print(f"    Primeiras linhas:")
                    print(df.head(2).to_string(index=False).replace('\n', '\n    '))
                    print()
                except Exception as e:
                    csv_info[file] = {
                        'path': path,
                        'status': '⚠️',
                        'erro': str(e)
                    }
                    print(f"⚠️  {path} - Erro: {e}\n")
    
    if not csv_files:
        print("  ⚠️  Nenhum CSV encontrado em resultados_1/. Execute as células de solução primeiro!")
        return {
            'status': 'erro',
            'mensagem': 'Nenhum CSV gerado',
            'arquivos': []
        }
    else:
        print(f"✅ Total de arquivos CSV gerados: {len(csv_files)}")
    
    print("\n" + "="*70)
    print("PRÓXIMAS ETAPAS")
    print("="*70)
    
    print("""
1. Abra o notebook: gerar_tabelas_latex.ipynb
2. Execute as células para gerar as tabelas LaTeX
3. Os CSVs acima serão automaticamente lidos e convertidos em tabelas

📁 Arquivos CSV para usar em gerar_tabelas_latex.ipynb:
""")
    
    for csv_path in sorted(csv_files):
        print(f"   - {csv_path}")
    
    print("""
Este fluxo garante que:
✓ Dados são calculados uma única vez (em solucao_coloracao_completa)
✓ CSVs armazenam os resultados (fonte de verdade única)
✓ Tabelas LaTeX são geradas automaticamente (sempre sincronizadas com dados)
✓ Comparação dos 3 algoritmos disponível para análise
✓ Arquivos com nomes descritivos: heuristica_Welsh-Powell.csv e heuristica_DSATUR.csv
""")
    
    return {
        'status': 'sucesso',
        'total_arquivos': len(csv_files),
        'arquivos': csv_files,
        'informacoes': csv_info
    }

print("✓ Função consolidar_e_salvar_dados_csv definida")

✓ Função consolidar_e_salvar_dados_csv definida


## 12. Salvamento de Dados e Visualizações

### Objetivo desta Seção
Esta seção foi **integrada** à função `executar_solucao_completa()`.

### O que acontece na execução completa:
1. **Salvamento de CSVs** - Dados salvos automaticamente:
   - `parametros_grafos.csv` (Parte 1)
   - `resultados_forca_bruta.csv` (Parte 1)
   - `heuristica_Welsh-Powell.csv` (Parte 2)
   - `heuristica_DSATUR.csv` (Parte 2)
   - `informacoes_instancias.csv` (Parte 2)
   - `comparacao_algoritmos.csv` (Parte 2)

2. **Geração de Gráficos** - Visualizações criadas automaticamente:
   - **Parte 1**: Gráficos de escalabilidade (tempo vs tamanho)
   - **Parte 2**: Gráficos comparativos das heurísticas

3. **Salvamento de Grafos** - PNGs coloridos:
   - **Parte 1**: 33 grafos aleatórios coloridos
   - **Parte 2**: 10 grafos DIMACS (5 Welsh-Powell + 5 DSATUR)

### Modularização
Toda a lógica de salvamento e visualização está **centralizada** na função `executar_solucao_completa()`, garantindo:
- ✅ Execução atômica (tudo ou nada)
- ✅ Consistência nos nomes de arquivos
- ✅ Validação automática de diretórios
- ✅ Relatório final consolidado

## 13. Função Principal: Execução Completa

### Objetivo desta Seção
Orquestrar toda a solução em uma única função centralizada que executa as 4 etapas principais em sequência com **3 ALGORITMOS**.

### 🎯 Por que Centralizar?
- **Modularidade**: Todas as etapas em um único ponto de entrada
- **Consistência**: Nomes de arquivos e estrutura padronizados
- **Manutenibilidade**: Fácil de entender o fluxo completo
- **Atomicidade**: Execução completa sem passos manuais intermediários

## Fluxo de Execução (Orquestração)

### Etapa 1: Criar Estrutura
- Cria diretórios para resultados
- Verifica e confirma criação
- Prepara ambiente para salvar arquivos

### Etapa 2: PARTE 1 - Força Bruta
**Tempo estimado**: 5-30 minutos dependendo do hardware
- Gera 27 instâncias aleatórias (5-13 vértices)
- Aplica algoritmo força bruta em cada
- Mede tempo de execução
- Salva 27 PNGs e CSVs
- **Demonstra**: Crescimento exponencial é real

### Etapa 3: PARTE 2 - Heurística
**Tempo estimado**: < 1 segundo
- Lê 5 arquivos DIMACS do Moodle
- Aplica Welsh-Powell (O(n+m))
- Registra cores encontradas
- Salva 5 CSVs + visualizações (se n≤500)
- **Demonstra**: Eficiência de heurísticas

### Etapa 4: Exportação CSV
- Consolida todos os resultados
- Cria 3 arquivos CSV (2 de Parte 1, 1 de Parte 2)
- Pronto para análise em Excel/Python

### Etapa 5: Geração de Gráficos
- 2 gráficos de Parte 1 (escalabilidade)
- 4 gráficos de Parte 2 (análise heurística)
- Todos salvos como PNG em alta resolução

## Resumo Final Exibido

### Estatísticas Parte 1
- Número total de instâncias processadas
- Intervalo de tamanhos testados
- Tempo máximo observado
- Tempo total agregado

### Estatísticas Parte 2
- Número de instâncias DIMACS
- Maior instância processada
- Tempo máximo (muito menor que Parte 1)

### Tempo Total
- Tempo de execução da solução completa
- Inclui todas as 5 etapas

### Localização de Arquivos
- Confirmação de todos os diretórios criados
- Facilita localização de resultados

In [17]:
def gerar_graficos_parte1(resultados_list: List):
    """
    Gera gráficos de análise para a Parte 1 (Força Bruta).
    Cria 2 imagens separadas: uma para Escalabilidade e uma para Número Cromático.
    
    Args:
        resultados_list: Lista de resultados da força bruta
    """
    if not resultados_list:
        print("⚠ Sem dados para gerar gráficos da Parte 1")
        return
    
    df = pd.DataFrame([{k: v for k, v in r.items() if k not in ['grafo', 'coloracao']} for r in resultados_list])
    
    # Agrupar por tamanho
    df_agrupado = df.groupby('tamanho').agg({
        'tempo_execucao': ['mean', 'min', 'max'],
        'num_cores': 'mean'
    }).reset_index()
    
    tamanhos = df_agrupado['tamanho']
    tempos_med = df_agrupado[('tempo_execucao', 'mean')]
    tempos_min = df_agrupado[('tempo_execucao', 'min')]
    tempos_max = df_agrupado[('tempo_execucao', 'max')]
    cores_med = df_agrupado[('num_cores', 'mean')]
    
    # Gráfico 1: Escalabilidade (Tempo vs Tamanho)
    fig1, ax1 = plt.subplots(figsize=(10, 6))
    ax1.semilogy(tamanhos, tempos_med, 'o-', color='blue', linewidth=2.5, markersize=10, label='Tempo Médio')
    ax1.fill_between(tamanhos, tempos_min, tempos_max, alpha=0.2, color='blue')
    ax1.set_xlabel('Número de Vértices', fontsize=13, fontweight='bold')
    ax1.set_ylabel('Tempo de Execução (s, escala log)', fontsize=13, fontweight='bold')
    ax1.set_title('Escalabilidade do Algoritmo de Força Bruta', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3, which='both')
    ax1.legend(fontsize=11)
    
    plt.tight_layout()
    caminho1 = ConfiguradorDiretorios.obter_caminho_arquivo(
        'parte1', 'graficos', 'escalabilidade_tempo.png'
    )
    fig1.savefig(caminho1, dpi=150, bbox_inches='tight')
    plt.close(fig1)
    print(f"✓ Gráfico de escalabilidade salvo: {caminho1}")
    
    # Gráfico 2: Número Cromático
    fig2, ax2 = plt.subplots(figsize=(10, 6))
    ax2.plot(tamanhos, cores_med, 's-', color='green', linewidth=2.5, markersize=10)
    ax2.set_xlabel('Número de Vértices', fontsize=13, fontweight='bold')
    ax2.set_ylabel('Número Cromático χ(G)', fontsize=13, fontweight='bold')
    ax2.set_title('Número Cromático vs Tamanho do Grafo', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    caminho2 = ConfiguradorDiretorios.obter_caminho_arquivo(
        'parte1', 'graficos', 'numero_cromatico.png'
    )
    fig2.savefig(caminho2, dpi=150, bbox_inches='tight')
    plt.close(fig2)
    print(f"✓ Gráfico de número cromático salvo: {caminho2}")


def gerar_graficos_parte2(resultados_wp: List, resultados_ds: List):
    """
    Gera gráficos de análise para a Parte 2 (Heurísticas).
    
    Args:
        resultados_wp: Resultados Welsh-Powell
        resultados_ds: Resultados DSATUR
    """
    if not resultados_wp or not resultados_ds:
        print("⚠ Sem dados para gerar gráficos da Parte 2")
        return
    
    df_wp = pd.DataFrame(resultados_wp)
    df_ds = pd.DataFrame(resultados_ds)
    
    # Gráfico de comparação
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Subplot 1: Comparação de cores
    x = range(len(df_wp))
    width = 0.35
    ax1.bar([i - width/2 for i in x], df_wp['cores_heuristica'], width, label='Welsh-Powell', color='skyblue')
    ax1.bar([i + width/2 for i in x], df_ds['cores_heuristica'], width, label='DSATUR', color='coral')
    ax1.set_xlabel('Instância', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Número de Cores', fontsize=12, fontweight='bold')
    ax1.set_title('Comparação: Welsh-Powell vs DSATUR', fontsize=13, fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels(df_wp['instancia_id'])
    ax1.legend(fontsize=10)
    ax1.grid(True, alpha=0.3, axis='y')
    
    # Subplot 2: Tempo de execução
    ax2.bar([i - width/2 for i in x], df_wp['tempo_segundos'], width, label='Welsh-Powell', color='skyblue')
    ax2.bar([i + width/2 for i in x], df_ds['tempo_segundos'], width, label='DSATUR', color='coral')
    ax2.set_xlabel('Instância', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Tempo (s)', fontsize=12, fontweight='bold')
    ax2.set_title('Tempo de Execução', fontsize=13, fontweight='bold')
    ax2.set_xticks(x)
    ax2.set_xticklabels(df_wp['instancia_id'])
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    caminho = ConfiguradorDiretorios.obter_caminho_arquivo(
        'parte2', 'graficos', 'comparacao_heuristicas.png'
    )
    plt.savefig(caminho, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Gráfico de análise Parte 2 salvo: {caminho}")


def gerar_graficos_parte2_detalhados(resultados_wp: List, resultados_ds: List):
    """
    Gera 4 gráficos detalhados para análise das heurísticas na Parte 2.
    Um conjunto de gráficos separados para cada heurística.
    
    Args:
        resultados_wp: Resultados Welsh-Powell
        resultados_ds: Resultados DSATUR
    """
    if not resultados_wp or not resultados_ds:
        print("⚠ Sem dados para gerar gráficos detalhados da Parte 2")
        return
    
    df_wp = pd.DataFrame(resultados_wp)
    df_ds = pd.DataFrame(resultados_ds)
    
    # Combinar dados com identificação de heurística
    df_wp['heuristica'] = 'Welsh-Powell'
    df_ds['heuristica'] = 'DSATUR'
    df_combined = pd.concat([df_wp, df_ds], ignore_index=True)
    
    # Gráfico 1: Cores por Instância (separado por heurística)
    fig1, ax1 = plt.subplots(figsize=(11, 6))
    x = range(len(df_wp))
    width = 0.35
    ax1.bar([i - width/2 for i in x], df_wp['cores_heuristica'], width, label='Welsh-Powell', color='skyblue')
    ax1.bar([i + width/2 for i in x], df_ds['cores_heuristica'], width, label='DSATUR', color='coral')
    ax1.set_xlabel('Instância', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Número de Cores', fontsize=12, fontweight='bold')
    ax1.set_title('Cores por Instância DIMACS', fontsize=13, fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels(df_wp['instancia_id'])
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    caminho1 = ConfiguradorDiretorios.obter_caminho_arquivo(
        'parte2', 'graficos', 'grafico1_cores_por_instancia.png'
    )
    fig1.savefig(caminho1, dpi=150, bbox_inches='tight')
    plt.close(fig1)
    print(f"✓ Gráfico 1 salvo: {caminho1}")
    
    # Gráfico 2: Tempo de Execução
    fig2, ax2 = plt.subplots(figsize=(11, 6))
    ax2.bar([i - width/2 for i in x], df_wp['tempo_segundos'], width, label='Welsh-Powell', color='skyblue')
    ax2.bar([i + width/2 for i in x], df_ds['tempo_segundos'], width, label='DSATUR', color='coral')
    ax2.set_xlabel('Instância', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Tempo (segundos)', fontsize=12, fontweight='bold')
    ax2.set_title('Tempo de Execução por Instância', fontsize=13, fontweight='bold')
    ax2.set_xticks(x)
    ax2.set_xticklabels(df_wp['instancia_id'])
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    caminho2 = ConfiguradorDiretorios.obter_caminho_arquivo(
        'parte2', 'graficos', 'grafico2_tempo_execucao.png'
    )
    fig2.savefig(caminho2, dpi=150, bbox_inches='tight')
    plt.close(fig2)
    print(f"✓ Gráfico 2 salvo: {caminho2}")
    
    # Gráfico 3: Vértices vs Cores
    fig3, ax3 = plt.subplots(figsize=(11, 6))
    ax3.scatter(df_wp['num_vertices'], df_wp['cores_heuristica'], s=150, alpha=0.7, 
                label='Welsh-Powell', color='skyblue', edgecolors='black', linewidth=1.5)
    ax3.scatter(df_ds['num_vertices'], df_ds['cores_heuristica'], s=150, alpha=0.7, 
                label='DSATUR', color='coral', edgecolors='black', linewidth=1.5)
    ax3.set_xlabel('Número de Vértices', fontsize=12, fontweight='bold')
    ax3.set_ylabel('Número de Cores', fontsize=12, fontweight='bold')
    ax3.set_title('Vértices vs Cores Utilizadas', fontsize=13, fontweight='bold')
    ax3.legend(fontsize=11)
    ax3.grid(True, alpha=0.3)
    plt.tight_layout()
    caminho3 = ConfiguradorDiretorios.obter_caminho_arquivo(
        'parte2', 'graficos', 'grafico3_vertices_vs_cores.png'
    )
    fig3.savefig(caminho3, dpi=150, bbox_inches='tight')
    plt.close(fig3)
    print(f"✓ Gráfico 3 salvo: {caminho3}")
    
    # Gráfico 4: Densidade vs Cores
    fig4, ax4 = plt.subplots(figsize=(11, 6))
    ax4.scatter(df_wp['densidade'], df_wp['cores_heuristica'], s=150, alpha=0.7, 
                label='Welsh-Powell', color='skyblue', edgecolors='black', linewidth=1.5)
    ax4.scatter(df_ds['densidade'], df_ds['cores_heuristica'], s=150, alpha=0.7, 
                label='DSATUR', color='coral', edgecolors='black', linewidth=1.5)
    ax4.set_xlabel('Densidade do Grafo', fontsize=12, fontweight='bold')
    ax4.set_ylabel('Número de Cores', fontsize=12, fontweight='bold')
    ax4.set_title('Densidade vs Cores Utilizadas', fontsize=13, fontweight='bold')
    ax4.legend(fontsize=11)
    ax4.grid(True, alpha=0.3)
    plt.tight_layout()
    caminho4 = ConfiguradorDiretorios.obter_caminho_arquivo(
        'parte2', 'graficos', 'grafico4_densidade_vs_cores.png'
    )
    fig4.savefig(caminho4, dpi=150, bbox_inches='tight')
    plt.close(fig4)
    print(f"✓ Gráfico 4 salvo: {caminho4}")

print("✓ Funções gerar_graficos_parte1 e gerar_graficos_parte2 definidas")


def gerar_csv_consolidado(df_fb: pd.DataFrame, df_wp: pd.DataFrame, df_ds: pd.DataFrame) -> pd.DataFrame:
    """
    Gera CSV consolidado comparando os 3 algoritmos (Força Bruta, Welsh-Powell, DSATUR).
    Salva em dois locais:
    - resultados_1/parte2/csv/comparacao_algoritmos.csv (usado internamente)
    - resultados_1/comparacao_algoritmos.csv (raiz para acesso fácil)
    
    Args:
        df_fb: DataFrame com resultados de Força Bruta
        df_wp: DataFrame com resultados de Welsh-Powell
        df_ds: DataFrame com resultados de DSATUR
        
    Returns:
        DataFrame consolidado com comparação dos 3 algoritmos
    """
    # Criar DataFrame de comparação
    df_comparacao = pd.DataFrame({
        'Algoritmo': ['Força Bruta', 'Welsh-Powell', 'DSATUR'],
        'Instâncias': [len(df_fb), len(df_wp), len(df_ds)],
        'Cores Médias': [
            f"{df_fb['num_cores'].mean():.2f}" if len(df_fb) > 0 else "N/A",
            f"{df_wp['cores_heuristica'].mean():.2f}" if len(df_wp) > 0 else "N/A",
            f"{df_ds['cores_heuristica'].mean():.2f}" if len(df_ds) > 0 else "N/A"
        ],
        'Cores Mín.': [
            int(df_fb['num_cores'].min()) if len(df_fb) > 0 else "N/A",
            int(df_wp['cores_heuristica'].min()) if len(df_wp) > 0 else "N/A",
            int(df_ds['cores_heuristica'].min()) if len(df_ds) > 0 else "N/A"
        ],
        'Cores Máx.': [
            int(df_fb['num_cores'].max()) if len(df_fb) > 0 else "N/A",
            int(df_wp['cores_heuristica'].max()) if len(df_wp) > 0 else "N/A",
            int(df_ds['cores_heuristica'].max()) if len(df_ds) > 0 else "N/A"
        ],
        'Tempo Médio (s)': [
            f"{df_fb['tempo_execucao'].mean():.4f}" if len(df_fb) > 0 else "N/A",
            f"{df_wp['tempo_segundos'].mean():.4f}" if len(df_wp) > 0 else "N/A",
            f"{df_ds['tempo_segundos'].mean():.4f}" if len(df_ds) > 0 else "N/A"
        ],
        'Qualidade': ['Ótima (χ exato)', 'Aproximada', 'Aproximada (melhor)'],
        'Complexidade': ['Exponencial', 'O(n²)', 'O(n²)']
    })
    
    # Salvar em parte2/csv (local padrão)
    caminho_parte2 = ConfiguradorDiretorios.obter_caminho_arquivo(
        'parte2', 'csv', 'comparacao_algoritmos.csv'
    )
    df_comparacao.to_csv(caminho_parte2, index=False)
    print(f"✓ Comparação salva em: {caminho_parte2}")
    
    # Salvar em raiz de resultados_1 (para acesso fácil)
    caminho_raiz = 'resultados_1/comparacao_algoritmos.csv'
    df_comparacao.to_csv(caminho_raiz, index=False)
    print(f"✓ Comparação salva em: {caminho_raiz}")
    
    return df_comparacao


print("✓ Função gerar_csv_consolidado definida")

✓ Funções gerar_graficos_parte1 e gerar_graficos_parte2 definidas
✓ Função gerar_csv_consolidado definida


In [ ]:
def executar_solucao_completa(tamanho_min: int = 5, tamanho_max: int = 15):
    """
    Orquestra a execução completa da solução com 4 etapas:
    1. FORÇA BRUTA: Análise em instâncias aleatórias (padrão: 5-15 vértices = 33 grafos)
    2. HEURÍSTICAS DIMACS: Welsh-Powell e DSATUR em 5 instâncias DIMACS
    3. COMPARAÇÃO: Consolidação dos 3 algoritmos para análise
    4. VALIDAÇÃO: Consolidar e salvar dados finais
    
    Args:
        tamanho_min: Tamanho mínimo dos grafos (padrão: 5)
        tamanho_max: Tamanho máximo dos grafos (padrão: 15, pode ir até 20)
    """
    
    from datetime import datetime
    hora_inicio = datetime.now()

    print("\n" + "="*80)
    print("🕐 INICIANDO EXECUÇÃO DA SOLUÇÃO COMPLETA")
    print("="*80)
    print(f"⏰ Data e Hora: {hora_inicio.strftime('%d/%m/%Y às %H:%M:%S')}")
    print(f"📊 Configuração: tamanho_min={tamanho_min}, tamanho_max={tamanho_max}")
    print(f"📈 Total de instâncias esperadas: {(tamanho_max - tamanho_min + 1) * 3 + 5} grafos")
    print("="*80)

    # ETAPA 1: FORÇA BRUTA
    print("\n[1/4] Executando Força Bruta em instâncias aleatórias...")
    print("-" * 80)
    
    instancias_aleatorias = gerar_instancias_aleatorias(tamanho_min, tamanho_max)
    print(f"✓ {len(instancias_aleatorias)} instâncias geradas")
    
    resultados_forca_bruta = processar_instancias_aleatorias(instancias_aleatorias)
    print(f"✓ Força bruta completada para {len(resultados_forca_bruta)} entradas")
    
    df_params = pd.DataFrame([r for r in resultados_forca_bruta if 'num_vertices' in r and 'num_cores' not in r])
    caminho_params = ConfiguradorDiretorios.obter_caminho_arquivo(
        'parte1', 'csv', 'parametros_grafos.csv'
    )
    df_params.to_csv(caminho_params, index=False)
    print(f"✓ Parâmetros salvos em: {caminho_params}")
    
    df_fb = pd.DataFrame([r for r in resultados_forca_bruta if 'num_cores' in r])
    caminho_fb = ConfiguradorDiretorios.obter_caminho_arquivo(
        'parte1', 'csv', 'resultados_forca_bruta.csv'
    )
    df_fb.to_csv(caminho_fb, index=False)
    print(f"✓ Resultados força bruta salvos em: {caminho_fb}")
    
    # Gerar gráficos de análise Parte 1
    print("\n  Gerando gráficos de análise...")
    resultados_list_parte1 = [r for r in resultados_forca_bruta if 'num_cores' in r]
    gerar_graficos_parte1(resultados_list_parte1)
    
    # ETAPA 2: HEURÍSTICAS (Welsh-Powell E DSATUR)
    print("\n[2/4] Executando Heurísticas em 5 instâncias DIMACS...")
    print("-" * 80)
    
    resultados_dimacs = processar_instancias_dimacs()
    print(f"✓ Processamento DIMACS concluído")
    
    # Salvar Welsh-Powell
    df_wp = pd.DataFrame(resultados_dimacs['resultados_welsh_powell'])
    caminho_wp = ConfiguradorDiretorios.obter_caminho_arquivo(
        'parte2', 'csv', 'heuristica_Welsh-Powell.csv'
    )
    df_wp.to_csv(caminho_wp, index=False)
    print(f"✓ Welsh-Powell salvo em: {caminho_wp}")
    
    # Salvar DSATUR
    df_ds = pd.DataFrame(resultados_dimacs['resultados_dsatur'])
    caminho_ds = ConfiguradorDiretorios.obter_caminho_arquivo(
        'parte2', 'csv', 'heuristica_DSATUR.csv'
    )
    df_ds.to_csv(caminho_ds, index=False)
    print(f"✓ DSATUR salvo em: {caminho_ds}")
    
    # Salvar informações DIMACS
    df_info = pd.DataFrame(resultados_dimacs['informacoes'])
    caminho_info = ConfiguradorDiretorios.obter_caminho_arquivo(
        'parte2', 'csv', 'informacoes_instancias.csv'
    )
    df_info.to_csv(caminho_info, index=False)
    print(f"✓ Informações instâncias DIMACS salvas em: {caminho_info}")
    
    # Gerar gráficos de análise Parte 2
    print("\n  Gerando gráficos de análise...")
    gerar_graficos_parte2(resultados_dimacs['resultados_welsh_powell'], resultados_dimacs['resultados_dsatur'])
    gerar_graficos_parte2_detalhados(resultados_dimacs['resultados_welsh_powell'], resultados_dimacs['resultados_dsatur'])
    
    # ETAPA 3: COMPARAÇÃO DOS 3 ALGORITMOS
    print("\n[3/4] Gerando relatório de comparação dos 3 algoritmos...")
    print("-" * 80)
    
    # Gerar CSV consolidado (modular)
    df_comparacao = gerar_csv_consolidado(df_fb, df_wp, df_ds)
    caminho_comp = 'resultados_1/comparacao_algoritmos.csv'
    print("\nResumo da Comparação:")
    print(df_comparacao.to_string(index=False))
    
    # ETAPA 4: CONSOLIDAÇÃO
    print("\n[4/4] Consolidando e validando dados...")
    print("-" * 80)
    resultado_consolidacao = consolidar_e_salvar_dados_csv()
    
    # RESUMO FINAL
    print("\n" + "="*80)
    print("RESUMO DA EXECUÇÃO COMPLETA")
    print("="*80)
    print(f"\n✓ FORÇA BRUTA (Parte 1):")
    print(f"  • Instâncias: {len(df_fb)} grafos aleatórios ({tamanho_min}-{tamanho_max} vértices)")
    print(f"  • Arquivo: {caminho_fb}")
    print(f"  • Cores encontradas: {int(df_fb['num_cores'].min())} a {int(df_fb['num_cores'].max())}")
    print(f"  • Tempo total: {df_fb['tempo_execucao'].sum():.2f}s")
    
    print(f"\n✓ WELSH-POWELL (Parte 2A - DIMACS):")
    print(f"  • Instâncias: {len(df_wp)} grafos DIMACS")
    print(f"  • Arquivo: {caminho_wp}")
    print(f"  • Cores encontradas: {int(df_wp['cores_heuristica'].min())} a {int(df_wp['cores_heuristica'].max())}")
    
    print(f"\n✓ DSATUR (Parte 2B - DIMACS):")
    print(f"  • Instâncias: {len(df_ds)} grafos DIMACS")
    print(f"  • Arquivo: {caminho_ds}")
    print(f"  • Cores encontradas: {int(df_ds['cores_heuristica'].min())} a {int(df_ds['cores_heuristica'].max())}")
    
    print(f"\n✓ COMPARAÇÃO CONSOLIDADA:")
    print(f"  • Arquivo: {caminho_comp}")
    print(f"  • Status: Comparação dos 3 algoritmos completa")
    
    print(f"\n✓ CONSOLIDAÇÃO:")
    if resultado_consolidacao['status'] == 'sucesso':
        print(f"  • Total de CSVs validados: {resultado_consolidacao['total_arquivos']}")
        print(f"  • Status: ✓ Todos os arquivos íntegros")
    else:
        print(f"  • Status: ⚠️ {resultado_consolidacao.get('mensagem', 'Erro')}")
    
    print("\n" + "="*80)
    print("FLUXO COMPLETO FINALIZADO COM SUCESSO!")
    print("="*80)
    
    hora_fim = datetime.now()
    tempo_total = hora_fim - hora_inicio

    print("\n" + "="*80)
    print("✅ EXECUÇÃO FINALIZADA")
    print("="*80)
    print(f"⏰ Horário de início:  {hora_inicio.strftime('%d/%m/%Y às %H:%M:%S')}")
    print(f"⏰ Horário de término: {hora_fim.strftime('%d/%m/%Y às %H:%M:%S')}")
    print(f"⏱️  Tempo total:        {str(tempo_total).split('.')[0]}")
    print(f"   ({tempo_total.total_seconds():.0f} segundos = {tempo_total.total_seconds()/60:.1f} minutos)")
    print("="*80)
    
    return {
        'forca_bruta': df_fb,
        'welsh_powell': df_wp,
        'dsatur': df_ds,
        'comparacao': df_comparacao,
        'consolidacao': resultado_consolidacao,
        'hora_inicio': hora_inicio,
        'hora_fim': hora_fim,
        'tempo_total': tempo_total
    }

✓ Função executar_solucao_completa definida
  Use: executar_solucao_completa() para 5-15 vértices
  Use: executar_solucao_completa(tamanho_max=18) para grafos maiores


## 14. Executar a Solução Completa

### Objetivo desta Seção
Orquestrar toda a solução em uma única função que executa as 4 etapas principais em sequência com **3 ALGORITMOS**.

## 🎯 Configuração

**Intervalo padrão**: 5-15 vértices (33 instâncias, 3 por tamanho)  
**Ajustável até**: 20 vértices (se o tempo permitir)

## Fluxo de Execução (Orquestração)

### Etapa 1: FORÇA BRUTA (Parte 1)
**Tempo estimado**: 5-30 minutos (depende do intervalo)
- Gera instâncias aleatórias (padrão: 5-15 vértices = **33 grafos**)
- Aplica algoritmo força bruta em cada
- Mede tempo de execução
- Salva CSVs e visualizações
- **Demonstra**: Crescimento exponencial é real

### Etapa 2: HEURÍSTICAS (Parte 2)
**Tempo estimado**: < 5 segundos
- Lê 5 arquivos DIMACS do Moodle
- Aplica **Welsh-Powell** (O(n²) - greedy por grau)
- Aplica **DSATUR** (O(n²) - greedy por saturação)
- Registra cores encontradas de AMBAS as heurísticas
- Salva 2 CSVs separados:
  - `heuristica_Welsh-Powell.csv`
  - `heuristica_DSATUR.csv`
- **Demonstra**: Eficiência de heurísticas e comparação entre elas

### Etapa 3: COMPARAÇÃO (Consolidação)
- Consolida resultados dos **3 algoritmos**
- Cria arquivo `comparacao_algoritmos.csv`
- Compara: Força Bruta vs Welsh-Powell vs DSATUR
- Gera gráficos de análise

### Etapa 4: VALIDAÇÃO (Consolidação CSV)
- Verifica integridade de todos os CSVs
- Lista arquivos gerados
- Pronto para análise em Excel/LaTeX

## 📁 Estrutura de Saída

```
resultados_1/
├── parte1/           (Força Bruta - 33 instâncias)
│   ├── csv/
│   │   ├── parametros_grafos.csv
│   │   └── resultados_forca_bruta.csv
│   ├── grafos/       (33 visualizações)
│   └── graficos/     (Gráficos de escalabilidade)
└── parte2/           (Heurísticas DIMACS - 5 instâncias)
    ├── csv/
    │   ├── heuristica_Welsh-Powell.csv    ⬅️ Welsh-Powell
    │   ├── heuristica_DSATUR.csv          ⬅️ DSATUR
    │   ├── informacoes_instancias.csv
    │   └── comparacao_algoritmos.csv     (3 algoritmos)
    ├── grafos/       (10 visualizações: 5 WP + 5 DSATUR)
    └── graficos/     (Gráficos comparativos)
```

## ⏱️ Tempo de Execução Estimado

**Por intervalo de tamanho:**
- **5-10 vértices**: Segundos (< 1 min)
- **5-13 vértices**: Minutos (1-5 min)
- **5-15 vértices**: Minutos (5-20 min)
- **5-18 vértices**: Dezenas de minutos (30-90 min)
- **5-20 vértices**: Horas (pode ser inviável)

## Resumo Final Exibido

### Estatísticas Parte 1 (Força Bruta)
- Número total de instâncias processadas
- Intervalo de tamanhos testados
- Tempo máximo observado
- Tempo total agregado

### Estatísticas Parte 2A (Welsh-Powell)
- 5 instâncias DIMACS processadas
- Cores encontradas por instância
- Tempo de execução (muito rápido)

### Estatísticas Parte 2B (DSATUR)
- 5 instâncias DIMACS processadas
- Cores encontradas por instância
- Comparação com Welsh-Powell

### Tempo Total
- Tempo de execução da solução completa
- Inclui todas as 4 etapas

### Localização de Arquivos
- Confirmação de todos os diretórios criados
- Facilita localização de resultados

## 🔧 Como Ajustar o Intervalo

```python
# Padrão (5-15 vértices)
executar_solucao_completa()

# Personalizado (5-18 vértices)
executar_solucao_completa(tamanho_max=18)

# Maior alcance (5-20 vértices) - MUITO DEMORADO!
executar_solucao_completa(tamanho_max=20)
```


In [ ]:
executar_solucao_completa(tamanho_max=20)


SOLUÇÃO COMPLETA - COLORAÇÃO DE GRAFOS (3 ALGORITMOS)
Configuração: Grafos aleatórios de 5 a 20 vértices
Total de instâncias Parte 1: 48 grafos

[1/4] Executando Força Bruta em instâncias aleatórias...
--------------------------------------------------------------------------------

📊 Gerando instâncias aleatórias: 5-20 vértices
   Total esperado: 48 instâncias

✓ 48 instâncias geradas

🔄 Processando 48 instâncias com Força Bruta...
⚠️  ATENÇÃO: Grafos maiores podem levar MUITO tempo!

  [1/48] Tamanho 5: χ(G)=2, tempo=0.0000s ✓
  [2/48] Tamanho 5: χ(G)=2, tempo=0.0000s ✓
  [3/48] Tamanho 5: χ(G)=2, tempo=0.0000s ✓
  [4/48] Tamanho 6: χ(G)=2, tempo=0.0000s ✓
  [5/48] Tamanho 6: χ(G)=2, tempo=0.0000s ✓
  [6/48] Tamanho 6: χ(G)=2, tempo=0.0000s ✓
  [7/48] Tamanho 7: χ(G)=4, tempo=0.0001s ✓
  [8/48] Tamanho 7: χ(G)=3, tempo=0.0000s ✓
  [9/48] Tamanho 7: χ(G)=2, tempo=0.0000s ✓
  [10/48] Tamanho 8: χ(G)=4, tempo=0.0001s ✓
  [11/48] Tamanho 8: χ(G)=4, tempo=0.0001s ✓
  [12/48] Tamanho 8: χ

{'forca_bruta':     id_instancia  tamanho  num_vertices  num_arestas  num_cores  \
 0              1        5             5            1          2   
 1              2        5             5            1          2   
 2              3        5             5            3          2   
 3              4        6             6            4          2   
 4              5        6             6            3          2   
 5              6        6             6            3          2   
 6              7        7             7           13          4   
 7              8        7             7           11          3   
 8              9        7             7            3          2   
 9             10        8             8           14          4   
 10            11        8             8           12          4   
 11            12        8             8            7          3   
 12            13        9             9           19          4   
 13            14        9       